# Task 02 — Transform

**Workshop: Final Pipeline | Stage 2 of 3**

## Goal

Read `bronze.lab_orders`, apply cleaning and enrichment transformations, write the result as `bronze.lab_orders_clean`.

```
bronze.lab_orders
        │
        ▼  cast · withColumn · when/otherwise · filter
        │
        ▼
bronze.lab_orders_clean
```

## Transformations to implement

| # | Transformation | Description |
|---|---------------|-------------|
| 1 | `cast` | `order_datetime` (String) → `TimestampType` |
| 2 | `withColumn` | Add `net_amount` = `total_amount` × (1 − `discount_percent` / 100) |
| 3 | `when / otherwise` | Add `order_tier`: `"Premium"` ≥ 500 · `"Standard"` ≥ 200 · `"Small"` |
| 4 | `filter` | Remove rows where `order_id` IS NULL |

> Column `net_amount` will be used by Task 03 for revenue aggregation.


In [ ]:
%run ../../setup/00_setup

In [ ]:
# ── Widgets ──
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{bronze_schema}.lab_orders_clean"

print(f"Source : {source_table}")
print(f"Target : {target_table}")


---

## Exercise 1 — Imports

Import all PySpark functions you need: `col`, `when`, `round`, `to_timestamp`.


In [ ]:
# 💡 HINT — pyspark.sql.functions:
#
#   from pyspark.sql.functions import col, when, round, to_timestamp
#   from pyspark.sql.types import TimestampType
#
#   Key functions used in this task:
#     col("name")                  → Column reference
#     to_timestamp(col, format)    → parse string to Timestamp
#                                    format: "yyyy-MM-dd'T'HH:mm:ss"
#     when(condition, value)       → conditional expression (like SQL CASE WHEN)
#         .when(condition, value)  → additional condition
#         .otherwise(value)        → default value (like SQL ELSE)
#     round(col, scale)            → round to N decimal places

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

---

## Exercise 2 — Read source table

Read the Bronze table `source_table` into a DataFrame.


In [ ]:
# 💡 HINT — reading a managed Delta table:
#
#   df = spark.table("catalog.schema.table_name")
#
#   Equivalent SQL:  SELECT * FROM catalog.schema.table_name
#   No path needed — Unity Catalog manages the location.

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify
print(f"Source rows : {orders_bronze.count():,}")
orders_bronze.printSchema()


---

## Exercise 3 — Apply transformations

Chain all 4 transformations listed in the Goal table into a single DataFrame.  
Name the result `orders_clean`.


In [ ]:
# 💡 HINT — chaining transformations:
#
#   orders_clean = (
#       df
#       # 1. cast order_datetime String → TimestampType
#       .withColumn("order_datetime",
#           to_timestamp(col("order_datetime"), "format_string"))
#
#       # 2. add net_amount (arithmetic expression)
#       .withColumn("net_amount",
#           round(col("col_a") * (lit(1) - col("col_b") / lit(100)), 2))
#
#       # 3. add order_tier with when/otherwise
#       .withColumn("order_tier",
#           when(col("amount_col") >= threshold_1, "Label1")
#           .when(col("amount_col") >= threshold_2, "Label2")
#           .otherwise("Label3"))
#
#       # 4. remove nulls
#       .filter(col("key_col").isNotNull())
#   )
#
#   Note: lit(value) wraps a Python literal as a Column expression.
#   from pyspark.sql.functions import lit

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify
print(f"Rows after transform : {orders_clean.count():,}")
orders_clean.select(
    "order_id", "order_datetime", "total_amount",
    "discount_percent", "net_amount", "order_tier"
).show(10, truncate=False)


---

## Exercise 4 — Write to Delta

Write `orders_clean` to `target_table` in overwrite mode.


In [ ]:
# 💡 HINT — write with overwriteSchema:
#
#   df.write
#       .format("delta")
#       .mode("overwrite")
#       .option("overwriteSchema", "true")
#       .saveAsTable("catalog.schema.table")
#
#   overwriteSchema=true is required when the new DataFrame
#   has more columns than the existing table (e.g. net_amount, order_tier).

# YOUR CODE HERE
raise NotImplementedError("Complete this cell before proceeding")

In [ ]:
# Verify
row_count = spark.table(target_table).count()
print(f"✅ {target_table}  →  {row_count:,} rows")
display(spark.table(target_table).limit(5))


In [ ]:
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "table": target_table,
    "rows": row_count
}))
